<a href="https://colab.research.google.com/github/mridul1592/aiml-practice-capstone/blob/main/FAS_poc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# --- Step 1: Install necessary libraries (if not already installed) ---
!pip install -qU  langchain langchain-community unstructured chromadb tiktoken sentence-transformers langchain-text-splitters
!pip install "unstructured[pdf]"
!sudo apt-get install -y poppler-utils

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 27.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 32.6 MB/s eta 0:00:00

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
^C


### Mount Google Drive

To access files directly from your Google Drive, you need to mount it first. This will prompt you to authorize Colab to access your Google Drive files.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Set `DOCS_FOLDER` to the correct Google Drive path

After mounting, you need to update the `DOCS_FOLDER` variable in the next cell to the actual path of your 'Agri_docs' folder within Google Drive. For example, if your folder is directly under 'My Drive', it would be `/content/drive/MyDrive/Agri_docs`.

In [2]:
# IMPORTANT: Update this path to your actual 'Agri_docs' folder in Google Drive
DOCS_FOLDER = '/content/drive/MyDrive/Colab Notebooks/Sem_3/AIML/Agri_docs' # Example path

print(f"DOCS_FOLDER set to: {DOCS_FOLDER}")

DOCS_FOLDER set to: /content/drive/MyDrive/Colab Notebooks/Sem_3/AIML/Agri_docs


In [ ]:
import os
from langchain_community.document_loaders import DirectoryLoader, UnstructuredFileLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter # Changed import path
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# DOCS_FOLDER is now defined in cell ozwlf7wOsCz5

# --- Step 2: Load documents from the specified folder ---
print(f"Loading documents from '{DOCS_FOLDER}'...")

# Ensure the directory exists
if not os.path.exists(DOCS_FOLDER):
    print(f"Error: The folder '{DOCS_FOLDER}' does not exist. Please create it and place your documents inside.")
    # You might want to create a dummy file for demonstration or exit
    # os.makedirs(DOCS_FOLDER, exist_ok=True)
    # with open(os.path.join(DOCS_FOLDER, "sample.txt"), "w") as f:
    #     f.write("This is a sample document about agriculture.")
    # print("Created a sample document for testing.")
    # Re-run the cell after creating the folder and documents.
else:
    try:
        # Use DirectoryLoader with UnstructuredFileLoader to handle various document types
        loader = DirectoryLoader(
            DOCS_FOLDER,
            glob="**/*",  # Load all files and subdirectories
            loader_cls=UnstructuredFileLoader,
            show_progress=True
        )
        documents = loader.load()
        print(f"Loaded {len(documents)} documents.")

        if not documents:
            print("No documents found in the specified folder.")
        else:
            # --- Step 3: Split documents into smaller chunks ---
            print("Splitting documents into chunks...")
            # For semantic splitting, RecursiveCharacterTextSplitter is a good general choice.
            # You can tune chunk_size and chunk_overlap to maintain semantic coherence.
            # For more advanced semantic chunking, libraries like LangChain's SemanticChunker could be used,
            # but they often require more setup and potentially different dependencies.
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=1000, # Adjust based on the average size of semantically coherent blocks
                chunk_overlap=200 # Provides context between chunks
            )
            texts = text_splitter.split_documents(documents)
            print(f"Split into {len(texts)} chunks.")

            # --- Step 4: Create embeddings ---
            print("Creating embeddings... This may take a while.")
            # Using HuggingFaceEmbeddings for local embeddings
            # 'sentence-transformers/all-mpnet-base-v2' is a commonly recommended model for general-purpose embeddings,
            # offering a good balance of performance and quality for various semantic tasks.
            embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

            # --- Step 5: Store embeddings in a vector store (Chroma) ---
            print("Storing embeddings in Chroma vector store...")
            # Create a Chroma vector store from the document chunks and embeddings
            # persist_directory can be set to save the vector store to disk
            persist_directory = "./chroma_db"
            vectordb = Chroma.from_documents(
                documents=texts,
                embedding=embeddings,
                persist_directory=persist_directory
            )
            vectordb.persist()
            print(f"Vector store created and saved to '{persist_directory}'.")
            print("Data preparation for RAG system complete!")
            print("You can now use 'vectordb' for similarity search in your RAG pipeline.")

    except Exception as e:
        print(f"An error occurred during document processing: {e}")
        print("Please ensure 'Agri_docs' folder contains supported document types (e.g., .txt, .pdf, .docx).")

Loading documents from '/content/drive/MyDrive/Colab Notebooks/Sem_3/AIML/Agri_docs'...


 17%|█▋        | 2/12 [00:38<02:50, 17.06s/it]